In [ ]:
!pip install python-dotenv pandas numpy
!pip install langchain langchain-community langchain-huggingface langchain-pinecone langchain-core
!pip install sentence-transformers gpt4all
!pip install transformers huggingface-hub
!pip install pandas numpy sentence-transformers
!pip install requests -U langchain-huggingface
!pip install crewai
!pip install crewai-tools
!pip install cohere langchain
!pip install langchain-cohere

In [20]:
!pip install -q streamlit jupyter-dash pyngrok

In [ ]:
!pip uninstall -y pinecone-client

In [ ]:
!pip uninstall -y langchain
!pip install langchain-huggingface

In [ ]:
!pip install --upgrade langchain
!pip install sentence-transformers

In [24]:
import os

# Replace these with your actual API keys
os.environ["COHERE_API_KEY"] = "kj6qTQTrdGKlviXr2N7pVCHMxI3Z1dG2avL1mqwq"
os.environ["TAVILY_API_KEY"] = "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"
os.environ["PINECONE_API_KEY"] = "pcsk_6L8ZmY_fh3AkwzUNVg87rneaKU2HgoApGK91wJJoVoYtX4HSdnKdxaWpuktSyjkApsBeZ"

In [33]:
%%writefile app.py
import os
import numpy as np
from uuid import uuid4
import litellm
# LANGCHAIN
from langchain_community.llms import GPT4All
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain.chains import RetrievalQA
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import LLMChain
from langchain_community.embeddings import HuggingFaceEmbeddings
# VECTOR STORE
import pinecone
# AGENTS
from langchain_cohere import ChatCohere
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import AgentExecutor, Tool
from langchain.agents.react.agent import create_react_agent
from langchain import hub
from crewai import Agent, Crew, Task
from langchain_huggingface import HuggingFaceEndpoint
from langchain.agents import Tool
from langchain_pinecone import PineconeVectorStore
from langchain_community.tools.tavily_search import TavilySearchResults
from crewai.tools import tool
import streamlit as st
# Initialize LLM
llm = ChatCohere(
    model="command-r",
    api_key=os.environ["COHERE_API_KEY"],
    temperature=0.7,
    max_tokens=512
)

# Initialize vector store and embeddings
index_name = "agenticragmodel"
embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = PineconeVectorStore(
    index_name=index_name,
    namespace="main",
    embedding=embed
)

# Memory for conversation
conversational_memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=1,
    return_messages=True
)

# Set up the RetrievalQA chain
qa_db = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

# Initialize Tavily search
tavily = TavilySearchResults(max_results=10, tavily_api_key=os.getenv("TAVILY_API_KEY"))

# Define tools for the agents
@tool("pinecone-document-store")
def pinecone_tool(query: str) -> str:
    """Searches Pinecone vector DB using the RetrievalQA chain."""
    return qa_db.run(query)

@tool("tavily-web-search")
def tavily_tool(query: str) -> str:
    """Searches the web using Tavily for up-to-date information."""
    return tavily.run(query)

# Define the Research Agent
research_agent = Agent(
    role="Research Analyst",
    goal="Search and analyze information from TED talks using ONLY the available tools, first search in Pinecone for resource; if not found then search it using Tavily",
    backstory="Expert in TED talk analysis with access to web search and document memory.",
    tools=[pinecone_tool, tavily_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    use_tools=True
)

# Define the Review Agent
review_agent = Agent(
    role="Content Verifier",
    goal="Verify and validate summaries using only the Pinecone document store.",
    backstory="Experienced verifier who ensures summaries are factual.",
    tools=[pinecone_tool],
    llm=llm,
    verbose=True,
    use_tools=True
)

# Streamlit UI
st.set_page_config(page_title="Research Agent Chatbot", layout="wide")
st.title("Research Agent Chatbot")
st.markdown("### Ask your question below and get a structured response.")

# Get user input via Streamlit text input
user_question = st.text_input("Enter your question:")

if st.button("Submit"):
    if user_question:
        # Define the task with the user's question and expected output structure
        task = Task(
            description=user_question,
            expected_output=(
                "1. Provide a clear title expressing your view on the current question.\n"
                "2. Give a concise 2-sentence summary.\n"
                "3. List the sources or citations used and their respective links.\n"
                "4. Use Pinecone (pinecone-document-store) only if the question includes TEDx or Academic references; otherwise, use Tavily.\n"
                "5. Include a sub-heading that indicates the tool used to obtain the source (either 'Pinecone' or 'Tavily').\n"
                "6. If Pinecone is used as the source, the sub-heading should read 'Pinecone' (and vice versa for Tavily).\n"
                "7. When no relevant information is found in the pinecone-document-store, default to using Tavily as the source.\n"
            ),
            agent=research_agent
        )

        # Create the Crew with both agents
        crew = Crew(
            agents=[research_agent, review_agent],
            tasks=[task],
            manager_llm=llm
        )

        st.info("Processing your request...")
        try:
            # Kick off the crew to get the result
            result = crew.kickoff()
            st.success("🧠 Final Result:")
            st.markdown(f"```\n{result}\n```")
        except Exception as e:
            st.error(f"An error occurred: {e}")
    else:
        st.warning("Please enter a question.")

Overwriting app.py


In [34]:
!npm install -g localtunnel
# Run Streamlit and LocalTunnel
!streamlit run app.py &>./logs.txt &
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇⠙⠹⠸⠼⠴your url is: https://two-lemons-push.loca.lt
^C


In [17]:
!cat /content/logs.txt | grep 'Tunnel Password'